# Carbon Taxes and CO2 Emissions: A Synthetic-Control Analysis in Python

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmg777/starter-academic-v501/blob/master/content/post/python_sc_co2tax/notebook.ipynb)

Runnable companion to the blog post
[*Carbon Taxes and CO2 Emissions*](https://carlos-mendez.org/post/python_sc_co2tax/).

**Inspired by** the [RTutor problem set](https://github.com/TheresaGraefe/RTutorCarbonTaxesAndCO2Emissions)
by Theresa Graefe (2020), which replicates
[Andersson (2019), *AEJ: Economic Policy* 11(4)](https://doi.org/10.1257/pol.20170144).
All empirical findings — datasets, donor pool, synthetic-control design, OLS/IV specifications — are Andersson's.

## The question we are answering

In **1991**, Sweden put a price on carbon dioxide. The reform combined three things — a new **carbon tax**, a new **VAT** on transport fuel (1990), and a partial cut to a pre-existing **energy tax**. Thirty years later we ask:

1. **Did the carbon tax actually reduce CO2 emissions from transport?**
2. **Did it cost Sweden any economic growth?**

## What is causal inference, and why is it hard?

A *causal* claim says "X caused Y to change". A *correlation* says "X and Y moved together". The two are very different.

Sweden's CO2 emissions fell after 1991. But lots of other things also changed: oil prices, car technology, recessions. To say the carbon tax *caused* the fall, we need to compare actual Sweden to a Sweden that **never had the carbon tax** — a **counterfactual**.

We can never observe that counterfactual directly. Causal inference is the craft of building a plausible one from real data. This notebook walks through three attempts, from worst to best:

| Method | Counterfactual | Main weakness |
|---|---|---|
| Naive pre/post in Sweden | Sweden before 1990 | Confounded by everything else over time |
| Difference-in-differences (DiD) | One control country (Denmark), or OECD average | Assumes parallel trends |
| Synthetic control (SCM) | Weighted blend of donor countries | Needs a good pre-period fit |

After estimating the headline effect, we validate it with **three placebo tests**, rule out **GDP** as a confounder, then turn to **OLS and IV regressions** to understand the demand-side mechanism.

## 1. Install dependencies

Run this cell once per Colab session. Already-installed packages are skipped.

We use four specialised packages on top of pandas / numpy / matplotlib:

- [`pysyncon`](https://sdfordham.github.io/pysyncon/) — synthetic-control estimator
- [`pyfixest`](https://pyfixest.org/) — OLS and instrumental-variable regressions
- [`statsmodels`](https://www.statsmodels.org/) — Newey–West HAC standard errors
- [`pyreadr`](https://github.com/ofajardo/pyreadr) — read R `.Rds` files from Python

In [ ]:
%pip install -q pyfixest pysyncon pyreadr statsmodels pandas numpy matplotlib

## 2. Setup, imports, and dark theme

We set the matplotlib `rcParams` once for a dark navy theme that matches the site's palette. Every figure inherits it automatically. The dark background also makes small treatment gaps more visible than a white background does.

In [ ]:
from __future__ import annotations
from pathlib import Path
import io, urllib.request, tempfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyreadr
import statsmodels.api as sm
import pyfixest as pf
from pysyncon import Dataprep, Synth

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ── Site color palette ──────────────────────────────────────────────────────
STEEL_BLUE  = "#6a9bcc"
WARM_ORANGE = "#d97757"
NEAR_BLACK  = "#141413"
TEAL        = "#00d4c8"

DARK_NAVY  = "#0f1729"
GRID_LINE  = "#1f2b5e"
LIGHT_TEXT = "#c8d0e0"
WHITE_TEXT = "#e8ecf2"

plt.rcParams.update({
    "figure.facecolor": DARK_NAVY, "axes.facecolor": DARK_NAVY,
    "axes.edgecolor": DARK_NAVY, "axes.linewidth": 0,
    "axes.labelcolor": LIGHT_TEXT, "axes.titlecolor": WHITE_TEXT,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.spines.left": False, "axes.spines.bottom": False,
    "axes.grid": True, "grid.color": GRID_LINE,
    "grid.linewidth": 0.6, "grid.alpha": 0.8,
    "xtick.color": LIGHT_TEXT, "ytick.color": LIGHT_TEXT,
    "xtick.major.size": 0, "ytick.major.size": 0,
    "text.color": WHITE_TEXT, "font.size": 12,
    "legend.frameon": False, "legend.fontsize": 11,
    "legend.labelcolor": LIGHT_TEXT,
    "figure.edgecolor": DARK_NAVY,
    "savefig.facecolor": DARK_NAVY, "savefig.edgecolor": DARK_NAVY,
})

SLUG = "python_sc_co2tax"
ROOT = Path.cwd()  # Notebook-friendly; works on Colab and locally.

SAVE_KW = dict(dpi=150, bbox_inches="tight",
               facecolor=DARK_NAVY, edgecolor=DARK_NAVY, pad_inches=0.05)

def savefig(name: str) -> None:
    """Save and show the current figure."""
    plt.savefig(ROOT / f"{SLUG}_{name}.png", **SAVE_KW)
    plt.show()
    plt.close()

def hline(title: str) -> None:
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)

print("Setup complete. Figures will be saved to:", ROOT)


## 3. Data sources

We load **six datasets** directly from their GitHub raw URLs. No local files are needed.

| Dataset | What it holds | Used for |
| --- | --- | --- |
| `carbontax_data.dta` | OECD panel, 15 countries × 46 years | DiD and Synthetic Sweden |
| `descr_Sweden.Rds` | Sweden time series: prices, taxes, CO2, GDP | Descriptive plots and GDP-gap analysis |
| `GDP_data.Rds` | 13-country GDP panel | Synthetic-GDP exercise |
| `regression_data.Rds` | Sweden time series for elasticity model | OLS and IV regressions |
| `leave_one_out_data.dta` | Pre-computed leave-one-out series | Robustness plot |
| `disentangling_data.dta` | Three counterfactual emission paths | Carbon-tax-vs-VAT decomposition |

`pandas.read_stata` accepts URLs directly. `pyreadr.read_r` only accepts local file paths, so we stream the bytes into a temp file first.

In [ ]:
BASE_URL = (
    "https://raw.githubusercontent.com/cmg777/starter-academic-v501/master/"
    "content/post/python_sc_co2tax/references/RTutorCarbonTax-master/"
    "inst/ps/CarbonTaxesAndCO2Emissions/material"
)
URL = {fname: f"{BASE_URL}/{fname}" for fname in [
    "carbontax_data.dta",
    "disentangling_data.dta",
    "leave_one_out_data.dta",
    "descr_Sweden.Rds",
    "regression_data.Rds",
    "GDP_data.Rds",
]}

def read_rds(url: str) -> pd.DataFrame:
    """Stream an .Rds file from a URL and return its sole data frame."""
    with urllib.request.urlopen(url) as resp, \
         tempfile.NamedTemporaryFile(suffix=".Rds", delete=False) as tmp:
        tmp.write(resp.read())
        tmp_path = tmp.name
    df = pyreadr.read_r(tmp_path)[None].reset_index(drop=True)
    return df

for k, u in URL.items():
    print(f"  {k:<28s}  {u}")


## 4. Load the data

A **panel dataset** has multiple units (here, countries) observed across multiple time periods (here, years). The outcome of interest throughout is per-capita CO2 emissions from transport in metric tons.

Three numbers about the time window matter for everything that follows:

- **46 years total** (1960–2005), giving us a long history.
- **30 pre-treatment years** (1960–1989) to build the counterfactual.
- **16 post-treatment years** (1990–2005) to measure the effect.

In [ ]:
panel  = pd.read_stata(URL["carbontax_data.dta"])
loo    = pd.read_stata(URL["leave_one_out_data.dta"])
disent = pd.read_stata(URL["disentangling_data.dta"])

descr_sweden = read_rds(URL["descr_Sweden.Rds"])
gdp_data     = read_rds(URL["GDP_data.Rds"])
reg_data     = read_rds(URL["regression_data.Rds"])

print(f"panel (carbontax_data.dta): {panel.shape}, countries={panel['country'].nunique()}, years={panel['year'].min():.0f}-{panel['year'].max():.0f}")
print(f"descr_Sweden.Rds:          {descr_sweden.shape}")
print(f"GDP_data.Rds:              {gdp_data.shape}, countries={gdp_data['country'].nunique()}")
print(f"regression_data.Rds:       {reg_data.shape}, years={reg_data['year'].min():.0f}-{reg_data['year'].max():.0f}")
print(f"disentangling_data.dta:    {disent.shape}")
print(f"leave_one_out_data.dta:    {loo.shape}")

## 5. Descriptive overview — decomposing Sweden's gasoline price

A good causal study always starts with descriptive plots. We *look* at the policy variable (taxes), the outcome (CO2 emissions), and the obvious mechanism (fuel consumption) before running any model.

Sweden's retail gasoline price is the sum of four parts:

1. **Wholesale price** — set by world oil markets.
2. **Energy tax** — long-standing, predates the reform.
3. **Carbon tax** — new in 1991, scaled to CO2 content.
4. **VAT** — new on transport fuel in 1990.

The first chart layers all four components. The second sums them into the retail price.

In [ ]:
ds = descr_sweden.copy()
print(ds[["year", "VAT", "en_tax", "CO2_tax", "pw_real", "total_tax"]].head().to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5.4))
ax.plot(ds["year"], ds["pw_real"], color=STEEL_BLUE, lw=2.2, label="Real wholesale price")
ax.plot(ds["year"], ds["en_tax"], color=WARM_ORANGE, lw=2.0, label="Energy tax")
ax.plot(ds["year"], ds["CO2_tax"], color=TEAL, lw=2.0, label="Carbon tax")
ax.plot(ds["year"], ds["VAT"], color="#c179c8", lw=1.8, label="VAT")
ax.axvline(1990, color=LIGHT_TEXT, lw=0.8, ls=":")
ax.set_xlabel("Year")
ax.set_ylabel("Real price components (SEK / litre)")
ax.set_title("Sweden — gasoline price decomposition (1960–2005)")
ax.legend(loc="upper left", ncol=2)
savefig("gasoline_price_components")

fig, ax = plt.subplots(figsize=(9, 5.4))
retail = ds["pw_real"] + ds["total_tax"]
ax.plot(ds["year"], retail, color=TEAL, lw=2.4, label="Retail gasoline price (real)")
ax.plot(ds["year"], ds["total_tax"], color=WARM_ORANGE, lw=2.0, label="Total tax")
ax.plot(ds["year"], ds["pw_real"], color=STEEL_BLUE, lw=2.0, label="Real wholesale price")
ax.axvline(1990, color=LIGHT_TEXT, lw=0.8, ls=":")
ax.set_xlabel("Year")
ax.set_ylabel("SEK / litre (real)")
ax.set_title("Sweden — retail price = wholesale + total tax")
ax.legend(loc="upper left")
savefig("retail_price")

**What to take from these plots:**

- The carbon tax (teal) is brand new in 1991 and grows steadily.
- The energy tax (orange) actually *drops* — the reform was partly a **tax swap**, not a pure tax hike.
- The wholesale price (blue) is dominated by 1970s–80s oil shocks, not Swedish policy.
- By 2005 the carbon tax matches the energy tax in size. That fact matters later when we decompose the reform.

## 6. Descriptive overview — CO2 emissions and fuel consumption

Did the reform actually cut emissions? We plot two things side by side:

1. Sweden's CO2 from transport vs the OECD mean.
2. Per-capita gasoline and diesel consumption in Sweden.

The second plot matters because a CO2 reduction could come from two different channels:

- Drivers *drive less* (pure consumption drop), or
- Drivers *switch fuels* (gasoline to more-efficient diesel).

We also plot per-capita CO2 for *every* donor country, to see where Sweden sits in the distribution before the reform.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
axes[0].plot(ds["year"], ds["CO2_Sweden"], color=WARM_ORANGE, lw=2.2)
axes[0].plot(ds["year"], ds["CO2_OECD"], color=STEEL_BLUE, lw=2.0, ls="--", label="OECD sample mean")
axes[0].axvline(1990, color=LIGHT_TEXT, lw=0.8, ls=":")
axes[0].set_title("Per-capita CO$_2$ from transport")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Metric tons / capita")
axes[0].legend(["Sweden", "OECD sample mean"], loc="lower right")

axes[1].plot(ds["year"], ds["gas_cons"], color=TEAL, lw=2.2, label="Gasoline")
axes[1].plot(ds["year"], ds["diesel_cons"], color=WARM_ORANGE, lw=2.0, label="Diesel")
axes[1].axvline(1990, color=LIGHT_TEXT, lw=0.8, ls=":")
axes[1].set_title("Per-capita fuel consumption — Sweden")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("kg oil-equivalent / capita")
axes[1].legend(loc="lower right")

plt.suptitle("Outcomes around the 1990 Swedish energy-tax reform",
             color=WHITE_TEXT, fontsize=13)
savefig("co2_vs_consumption")

countries = sorted(panel["country"].unique())
fig, axes = plt.subplots(3, 5, figsize=(15, 8.5), sharex=True, sharey=True)
for ax, country in zip(axes.ravel(), countries):
    sub = panel[panel["country"] == country].sort_values("year")
    color = WARM_ORANGE if country == "Sweden" else STEEL_BLUE
    lw = 2.4 if country == "Sweden" else 1.4
    ax.plot(sub["year"], sub["CO2_transport_capita"], color=color, lw=lw)
    ax.axvline(1990, color=LIGHT_TEXT, lw=0.6, ls=":")
    ax.set_title(country, color=WHITE_TEXT, fontsize=11)
plt.suptitle("CO$_2$ from transport, per capita — OECD donor pool",
             color=WHITE_TEXT, fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.96])
savefig("co2_donor_pool")

**What to take from these plots:**

- Before 1990 Sweden's CO2 path tracks the OECD mean. After 1990 Sweden plateaus while the OECD keeps climbing — the first visible sign of an effect.
- Gasoline use peaks in the late 1980s and falls. Diesel grows steadily — part of the CO2 reduction is **fuel substitution**, not less driving.
- In the small-multiples grid, Sweden (orange) sits squarely in the middle of the donor distribution before 1990, neither highest nor lowest. Many donors are plausible matches.

## 7. Estimating causal effects — naive time-difference and DiD

We now move from looking at the data to *estimating* the policy's effect. We will try three estimators, ordered from worst to best.

### (a) Single-country before/after

The simplest model regresses Sweden's CO2 on a 0/1 post-1990 dummy:

$$\text{CO2}_{\text{Sweden},t} = \alpha + \delta \cdot \mathbf{1}\{t \geq 1990\} + \varepsilon_t$$

This treats every other thing that changed in Sweden between 1960 and 2005 (population, income, vehicle stock) as if it were part of the carbon tax's effect. **It will give the wrong answer.**

### (b) Difference-in-differences (DiD)

DiD compares Sweden's pre-vs-post change to a *control's* pre-vs-post change. The leftover is the "extra" change in Sweden, hopefully reflecting the treatment.

$$y_{jt} = \beta_0 + \beta_1 T_j + \beta_2 P_t + \beta_3 (T_j \cdot P_t) + \varepsilon_{jt}$$

where $T_j = 1$ if country $j$ is Sweden, $P_t = 1$ if $t \geq 1990$, and **$\beta_3$ is the DiD coefficient** (the average treatment effect on the treated, ATT). The key assumption: **parallel trends** — Sweden and the control would have moved in step without the reform.

In [ ]:
panel = panel.copy()
panel["post"] = (panel["year"] >= 1990).astype(int)
panel["treated"] = (panel["country"] == "Sweden").astype(int)
panel["Sweden_post"] = panel["treated"] * panel["post"]

# Time difference within Sweden alone (Sweden_time)
sw = panel[panel["country"] == "Sweden"].copy()
sw["delta"] = (sw["year"] >= 1990).astype(int)
m_time = pf.feols("CO2_transport_capita ~ delta", data=sw, vcov="HC1")
print("\nSweden time-difference regression:")
print(m_time.tidy().round(4))

# DiD on Sweden vs Denmark only
two = panel[panel["country"].isin(["Sweden", "Denmark"])].copy()
m_did2 = pf.feols(
    "CO2_transport_capita ~ treated + post + Sweden_post",
    data=two, vcov="HC1",
)
print("\nDiD: Sweden vs Denmark (HC1):")
print(m_did2.tidy().round(4))

# DiD on full OECD donor pool (clustered SE by country)
m_did_oecd = pf.feols(
    "CO2_transport_capita ~ treated + post + Sweden_post",
    data=panel, vcov={"CRV1": "country"},
)
print("\nDiD: Sweden vs full OECD donor pool (cluster by country):")
print(m_did_oecd.tidy().round(4))

# Save a tidy DiD comparison table
did_tab = pd.concat(
    [m_did2.tidy().assign(model="Sweden vs Denmark"),
     m_did_oecd.tidy().assign(model="Sweden vs OECD")],
    axis=0
)
did_tab.to_csv(ROOT / "tab_did_comparison.csv")

# Visual DiD: Sweden vs Denmark
fig, ax = plt.subplots(figsize=(9, 5.4))
for cty, col in [("Sweden", WARM_ORANGE), ("Denmark", STEEL_BLUE)]:
    sub = panel[panel["country"] == cty].sort_values("year")
    ax.plot(sub["year"], sub["CO2_transport_capita"], color=col, lw=2.2, label=cty)
ax.axvline(1990, color=LIGHT_TEXT, lw=0.8, ls=":")
ax.set_xlabel("Year")
ax.set_ylabel("Metric tons CO$_2$ / capita (transport)")
ax.set_title("DiD baseline — Sweden vs Denmark")
ax.legend(loc="lower right")
savefig("did_sweden_denmark")

**Reading the output:**

- **Naive Sweden time-difference:** +0.55 t/capita (t = 7.0). This says post-1990 Sweden emits *more* — because the window includes Sweden's growth through 2005. **Wrong answer, as expected.**
- **DiD Sweden vs Denmark:** −0.140 t/capita. Flips the sign! But underpowered (p = 0.23).
- **DiD Sweden vs OECD pool (cluster SE):** −0.214 t/capita (p = 0.02). Statistically significant.

Both DiD estimates are economically large (7–11% of Sweden's pre-reform level). But the parallel-trends assumption is not obviously satisfied for Sweden vs OECD before 1990. That motivates the next step: synthetic control.

## 8. Synthetic Sweden — the core idea + the math

### The idea
DiD picks a single control country (or unweighted average) as the counterfactual. That is rigid. What if no single country looks like Sweden, but a **blend** — 30% Denmark + 27% Belgium + 15% New Zealand + ... — does?

That blend is **Synthetic Sweden**. The synthetic-control method (SCM) picks the blend weights to make synthetic-Sweden match real Sweden as closely as possible *before* the reform. After the reform, the gap is the estimated treatment effect.

### The math, briefly

Let $X_1$ be Sweden's pre-treatment predictors (GDP per capita, vehicles, gasoline consumption, urbanisation, plus three lagged CO2 levels). Let $X_0$ be the same predictors for each donor.

Synthetic control picks donor weights $w$ to solve:

$$w^* = \arg\min_{w} (X_1 - X_0 w)^\top V (X_1 - X_0 w) \quad \text{s.t.} \quad w_j \geq 0, \; \sum_j w_j = 1$$

The matrix $V$ tells the optimiser how much weight to give each predictor. It is also chosen automatically to minimise the pre-period MSPE of the *outcome* (CO2). So there are two nested optimisations: pick $w$ given $V$, then pick $V$ to fit CO2 best.

### The code
`pysyncon` hides the nested optimisation behind two objects: `Dataprep(...)` packs the panel into the matrices, then `Synth().fit(...)` runs the optimisation. The cell also constructs Synthetic Sweden's emission path, computes the year-by-year gap, and saves the path / gap / weights plots.

In [ ]:
# Build with country names (pysyncon accepts unit names directly).
controls = [c for c in countries if c != "Sweden"]
dataprep = Dataprep(
    foo=panel,
    predictors=["GDP_per_capita", "vehicles_capita", "gas_cons_capita", "urban_pop"],
    predictors_op="mean",
    time_predictors_prior=range(1980, 1990),
    special_predictors=[
        ("CO2_transport_capita", [1989], "mean"),
        ("CO2_transport_capita", [1980], "mean"),
        ("CO2_transport_capita", [1970], "mean"),
    ],
    dependent="CO2_transport_capita",
    unit_variable="country",
    time_variable="year",
    treatment_identifier="Sweden",
    controls_identifier=controls,
    time_optimize_ssr=range(1960, 1990),
)

synth = Synth()
synth.fit(dataprep=dataprep, optim_method="Nelder-Mead", optim_initial="equal")

print("\nDonor weights for Synthetic Sweden (top 8):")
w_sorted = synth.weights().sort_values(ascending=False)
print(w_sorted.head(8).round(4))
print(f"\nWeights sum to {w_sorted.sum():.6f}")
v_labels = list(dataprep.predictors) + [f"{name}({yrs[0] if len(yrs)==1 else f'{yrs[0]}-{yrs[-1]}'})"
                                         for (name, yrs, _) in dataprep.special_predictors]
v_diag = np.diag(synth.V) if synth.V.ndim == 2 else np.asarray(synth.V).ravel()
print(f"V (predictor) weights:\n{pd.Series(v_diag, index=v_labels).round(4)}")
print(f"Loss V (MSPE pre): {synth.loss_V:.6f}")
print(f"Loss W:           {synth.loss_W:.6f}")

# Build Sweden actual vs synthetic series
years = np.arange(1960, 2006)
y_sweden = panel[panel["country"] == "Sweden"].set_index("year").loc[years, "CO2_transport_capita"]
panel_wide = panel.pivot(index="year", columns="country", values="CO2_transport_capita")
y_synth = panel_wide.loc[years, controls] @ w_sorted.reindex(controls).fillna(0)

actual_vs_synth = pd.DataFrame({"sweden": y_sweden.values,
                                "synth_sweden": y_synth.values,
                                "gap": y_sweden.values - y_synth.values},
                               index=years)
actual_vs_synth.index.name = "year"
actual_vs_synth.to_csv(ROOT / "tab_synth_sweden.csv")

print(f"\nSweden 2005: {y_sweden.loc[2005]:.4f} t/capita")
print(f"Synth Sweden 2005: {y_synth.loc[2005]:.4f} t/capita")
print(f"Gap 2005: {y_sweden.loc[2005] - y_synth.loc[2005]:.4f} t/capita "
      f"({(y_sweden.loc[2005]-y_synth.loc[2005])/y_synth.loc[2005]*100:.2f}% vs synth)")

post = (years >= 1990)
gap = y_sweden.values - y_synth.values
avg_post_pct = (gap[post] / y_synth.values[post]).mean() * 100
print(f"Average post-treatment gap (1990-2005): {gap[post].mean():.4f} t/capita ({avg_post_pct:.2f}%)")

# Path plot
fig, ax = plt.subplots(figsize=(9, 5.4))
ax.plot(years, y_sweden.values, color=WARM_ORANGE, lw=2.4, label="Sweden")
ax.plot(years, y_synth.values, color=STEEL_BLUE, lw=2.2, ls="--", label="Synthetic Sweden")
ax.axvline(1990, color=LIGHT_TEXT, lw=0.8, ls=":")
ax.set_xlabel("Year")
ax.set_ylabel("Metric tons CO$_2$ / capita (transport)")
ax.set_title("Path plot — Sweden vs Synthetic Sweden")
ax.legend(loc="lower right")
savefig("synth_sweden_fit")

# Weights bar
fig, ax = plt.subplots(figsize=(8.5, 5.4))
nonzero = w_sorted[w_sorted > 1e-4]
ax.barh(nonzero.index[::-1], nonzero.values[::-1], color=TEAL)
ax.set_xlabel("Donor weight in Synthetic Sweden")
ax.set_title("Country weights $w^*$ (Synthetic Sweden)")
for i, (lbl, v) in enumerate(zip(nonzero.index[::-1], nonzero.values[::-1])):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", color=LIGHT_TEXT, fontsize=10)
savefig("synth_weights")

# Gap plot
fig, ax = plt.subplots(figsize=(9, 5.4))
ax.plot(years, gap, color=WARM_ORANGE, lw=2.4)
ax.fill_between(years, gap, 0, where=(gap < 0), color=WARM_ORANGE, alpha=0.18)
ax.axvline(1990, color=LIGHT_TEXT, lw=0.8, ls=":")
ax.axhline(0, color=LIGHT_TEXT, lw=0.6)
ax.set_xlabel("Year")
ax.set_ylabel("Gap = Sweden − Synthetic Sweden (t CO$_2$ / cap)")
ax.set_title("Treatment gap in CO$_2$ from transport")
savefig("synth_gap")

w_sorted.to_csv(ROOT / "tab_synth_weights.csv")

**Reading the output:**

- **Donor weights:** Denmark (28.9%), Belgium (26.9%), New Zealand (14.6%), Greece (11.4%), USA (10.1%), Switzerland (7.9%). Same six donors Andersson picks in R. The other 9 donors get essentially 0%.
- **Weights sum to 1.000** — a proper convex combination, no extrapolation.
- **Pre-period fit:** Synthetic Sweden tracks real Sweden almost perfectly between 1960 and 1990 (small pre-MSPE).
- **Post-period gap:** by 2005 the gap is −0.36 t/capita (−15% relative to synthetic). The average post-treatment gap is **−0.27 t/capita per year, or −11.3%**.

In plain headline terms, the carbon tax (plus VAT) is associated with roughly **one ton of avoided per-capita transport CO2 every 3.7 years**. But is this real or noise? On to placebo tests.

## 9. Placebo tests — is this just noise?

The synthetic-control optimiser is *designed* to make Sweden look unique in the post-period. We need to check the gap is not an artefact of the method.

Three falsification tests, all run below:

| Test | What we do | What we expect if the effect is real |
|---|---|---|
| **In-time placebo** | Fit SCM as if reform happened in 1980 | No gap between 1980 and 1990 |
| **In-space placebos** | Re-run SCM 15 times, treating each donor as treated | Sweden's gap larger than most placebo gaps |
| **Leave-one-out** | Drop each high-weight donor one at a time | Result barely changes |

For the in-space test, we use the **post-/pre-treatment MSPE ratio** as the test statistic. The numerator is post-period deviation from the synthetic; the denominator is pre-period fit. Dividing penalises units whose synthetic was a poor fit to begin with.

In [ ]:
# (a) In-time placebo: reassign treatment to 1980
dp_time = Dataprep(
    foo=panel,
    predictors=["GDP_per_capita", "vehicles_capita", "gas_cons_capita", "urban_pop"],
    predictors_op="mean",
    time_predictors_prior=range(1970, 1980),
    special_predictors=[
        ("CO2_transport_capita", [1979], "mean"),
        ("CO2_transport_capita", [1970], "mean"),
        ("CO2_transport_capita", [1965], "mean"),
    ],
    dependent="CO2_transport_capita",
    unit_variable="country",
    time_variable="year",
    treatment_identifier="Sweden",
    controls_identifier=controls,
    time_optimize_ssr=range(1960, 1980),
)
synth_time = Synth()
synth_time.fit(dataprep=dp_time, optim_method="BFGS")
w_time = synth_time.weights()
yrs_time = np.arange(1960, 1991)
y_sw_t = panel_wide.loc[yrs_time, "Sweden"].values
y_sy_t = (panel_wide.loc[yrs_time, controls] @ w_time.reindex(controls).fillna(0)).values

fig, ax = plt.subplots(figsize=(9, 5.4))
ax.plot(yrs_time, y_sw_t, color=WARM_ORANGE, lw=2.4, label="Sweden")
ax.plot(yrs_time, y_sy_t, color=STEEL_BLUE, lw=2.2, ls="--", label="Synthetic Sweden")
ax.axvline(1980, color=LIGHT_TEXT, lw=0.8, ls=":")
ax.set_xlabel("Year")
ax.set_ylabel("Metric tons CO$_2$ / capita (transport)")
ax.set_title("In-time placebo — backdating treatment to 1980")
ax.legend(loc="lower right")
savefig("placebo_in_time")

# (b) In-space placebos: re-run SCM on every donor country
def run_placebo(treated_country: str) -> dict | None:
    co = [c for c in countries if c != treated_country]
    try:
        dp = Dataprep(
            foo=panel,
            predictors=["GDP_per_capita", "vehicles_capita", "gas_cons_capita", "urban_pop"],
            predictors_op="mean",
            time_predictors_prior=range(1980, 1990),
            special_predictors=[
                ("CO2_transport_capita", [1989], "mean"),
                ("CO2_transport_capita", [1980], "mean"),
                ("CO2_transport_capita", [1970], "mean"),
            ],
            dependent="CO2_transport_capita",
            unit_variable="country",
            time_variable="year",
            treatment_identifier=treated_country,
            controls_identifier=co,
            time_optimize_ssr=range(1960, 1990),
        )
        sy = Synth()
        sy.fit(dataprep=dp, optim_method="BFGS")
        w = sy.weights().reindex(co).fillna(0)
        y_actual = panel_wide.loc[years, treated_country].values
        y_syn = (panel_wide.loc[years, co] @ w).values
        gap_p = y_actual - y_syn
        pre_mask = years < 1990
        post_mask = years >= 1990
        mspe_pre = (gap_p[pre_mask] ** 2).mean()
        mspe_post = (gap_p[post_mask] ** 2).mean()
        return {"country": treated_country, "gap": gap_p,
                "mspe_pre": mspe_pre, "mspe_post": mspe_post,
                "ratio": mspe_post / mspe_pre if mspe_pre > 0 else np.nan}
    except Exception as e:
        print(f"  placebo failed for {treated_country}: {e}")
        return None


placebo_results = []
print("\nRunning in-space placebos for each donor country (this takes a moment)...")
for c in countries:
    r = run_placebo(c)
    if r is not None:
        placebo_results.append(r)

# Filter as Andersson/SCtools does: drop placebos with MSPE > 20 × Sweden's
sweden_res = next(r for r in placebo_results if r["country"] == "Sweden")
mspe_limit = 20 * sweden_res["mspe_pre"]
keep = [r for r in placebo_results
        if r["country"] == "Sweden" or r["mspe_pre"] <= mspe_limit]
print(f"Kept {len(keep)} placebos after MSPE-limit filter (limit = {mspe_limit:.4f}).")

fig, ax = plt.subplots(figsize=(9, 5.4))
for r in keep:
    if r["country"] == "Sweden":
        continue
    ax.plot(years, r["gap"], color=LIGHT_TEXT, lw=1.0, alpha=0.55)
ax.plot(years, sweden_res["gap"], color=WARM_ORANGE, lw=2.6, label="Sweden")
ax.axvline(1990, color=LIGHT_TEXT, lw=0.8, ls=":")
ax.axhline(0, color=LIGHT_TEXT, lw=0.6)
ax.set_xlabel("Year")
ax.set_ylabel("Gap in per-capita CO$_2$ from transport")
ax.set_title("In-space placebos — Sweden vs donor-pool placebos")
ax.legend(loc="lower left")
savefig("placebo_in_space")

# Post/pre MSPE ratio plot + p-value
ratios = pd.DataFrame(
    [{"country": r["country"], "ratio": r["ratio"]} for r in placebo_results]
).sort_values("ratio", ascending=False).reset_index(drop=True)
print("\nPost/Pre MSPE ratio per unit:")
print(ratios.round(3).to_string(index=False))
p_val = (ratios["ratio"] >= sweden_res["ratio"]).mean()
print(f"\nPermutation p-value for Sweden = {p_val:.4f}")
ratios.to_csv(ROOT / "tab_placebo_mspe_ratios.csv", index=False)

fig, ax = plt.subplots(figsize=(8.5, 6.0))
colors = [WARM_ORANGE if c == "Sweden" else STEEL_BLUE for c in ratios["country"]]
ax.barh(ratios["country"][::-1], ratios["ratio"][::-1], color=colors[::-1])
ax.set_xlabel("Post-/Pre-treatment MSPE ratio")
ax.set_title(f"Permutation test: MSPE ratios — p = {p_val:.3f}")
savefig("placebo_mspe_ratio")

# (c) Leave-one-out: use pre-computed dta file
fig, ax = plt.subplots(figsize=(9, 5.4))
excl_cols = [c for c in loo.columns if c.startswith("excl_")]
loo_long_labels = {
    "excl_unitedstates": "US excl.", "excl_belgium": "BE excl.",
    "excl_denmark": "DK excl.", "excl_greece": "GR excl.",
    "excl_newzealand": "NZ excl.", "excl_switzerland": "CH excl.",
}
for col in excl_cols:
    ax.plot(loo["Year"], loo[col], color=LIGHT_TEXT, lw=1.1, alpha=0.7,
            label=loo_long_labels.get(col, col))
ax.plot(loo["Year"], loo["synth_sweden"], color=WARM_ORANGE, lw=2.4,
        label="Synthetic Sweden")
ax.plot(loo["Year"], loo["sweden"], color=TEAL, lw=2.2, label="Sweden")
ax.axvline(1990, color=LIGHT_TEXT, lw=0.8, ls=":")
ax.set_xlabel("Year")
ax.set_ylabel("Metric tons CO$_2$ / capita (transport)")
ax.set_title("Leave-one-out robustness — Synthetic Sweden across donor exclusions")
ax.legend(loc="lower right", ncol=2, fontsize=9)
savefig("placebo_leave_one_out")

**All three tests pass:**

- **In-time placebo:** Sweden and Synthetic Sweden track each other through 1990. No false-positive gap. ✓
- **In-space placebos:** Sweden has the highest post/pre-MSPE ratio of any unit. Permutation **p = 0.067** (= 1/15, the smallest possible non-trivial p-value with 15 donors). ✓
- **Leave-one-out:** dropping each high-weight donor moves the estimated reduction only between 8.8% (without Switzerland) and 13% (without Denmark). All six exclusions still give a bigger effect than the DiD's 8.3%. ✓

The −11.3% reduction is unlikely to be an artefact of the method.

## 10. Was GDP a confounder?

A **confounder** is a variable that affects both the treatment and the outcome. The most common objection to the carbon-tax story: maybe Sweden's emissions fell for economic reasons (a recession, a structural shift) and the carbon tax just happened to coincide.

We rule this out in two steps. **Step 1:** compare CO2 and GDP gaps side by side.

If recessions drove the CO2 reduction, the CO2 gap should mirror the GDP gap (dip in recessions, rebound in recoveries). The shaded bands mark Sweden's two recessions: 1976–78 and 1991–93.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
axes[0].plot(ds["year"], ds["GDP_Sweden"], color=STEEL_BLUE, lw=2.2)
axes[0].axvline(1990, color=LIGHT_TEXT, lw=0.8, ls=":")
axes[0].set_title("GDP per capita — Sweden")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("USD per capita (real)")

axes[1].plot(ds["year"], ds["CO2_Sweden"], color=WARM_ORANGE, lw=2.2)
axes[1].axvline(1990, color=LIGHT_TEXT, lw=0.8, ls=":")
axes[1].set_title("CO$_2$ per capita — Sweden")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Metric tons / capita")
plt.suptitle("Sweden: GDP and CO$_2$ levels", color=WHITE_TEXT, fontsize=13)
savefig("gdp_co2_levels")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for ax, var, color, lab in [
    (axes[0], "gap_GDP", STEEL_BLUE, "Gap GDP (USD) — Sweden − Synth(CO$_2$)"),
    (axes[1], "gap_CO2", WARM_ORANGE, "Gap CO$_2$ (t / cap) — Sweden − Synth(CO$_2$)"),
]:
    ax.axvspan(1976, 1978, color=GRID_LINE, alpha=0.55)
    ax.axvspan(1991, 1993, color=GRID_LINE, alpha=0.55)
    ax.plot(ds["year"], ds[var], color=color, lw=2.2)
    ax.axhline(0, color=LIGHT_TEXT, lw=0.6)
    ax.set_xlabel("Year")
    ax.set_ylabel(lab)
plt.suptitle("Gaps vs Synthetic Sweden(CO$_2$) — recessions shaded",
             color=WHITE_TEXT, fontsize=13)
savefig("gdp_co2_gaps")

**What to look for:**

The GDP gap dips deeply during both recessions, then rebounds. **The CO2 gap dips during the 1991–93 recession but never rebounds.** Even when Swedish GDP fully recovered after 1993, emissions did not snap back. That asymmetry is the smoking gun against the "weak economy did it" story.

**Step 2:** build a second Synthetic Sweden, this time with GDP per capita as the outcome.

If the carbon tax depressed Swedish growth, actual GDP should fall *below* synthetic-GDP after 1990. If the two paths overlap, no growth penalty.

In [ ]:
gdp = gdp_data.copy()
# Some rows may have NaNs in predictors; pysyncon will fail if predictor is NaN
# for the years used. We drop schooling NaNs only for the years we average.
gdp_controls = sorted([c for c in gdp["country"].unique() if c != "Sweden"])

dp_gdp = Dataprep(
    foo=gdp,
    predictors=["investrate", "trade", "infrate"],
    predictors_op="mean",
    time_predictors_prior=range(1980, 1990),
    special_predictors=[
        ("gdp_cap", [1975], "mean"),
        ("gdp_cap", [1980], "mean"),
        ("gdp_cap", [1989], "mean"),
        ("schooling", [1975, 1980, 1985], "mean"),
    ],
    dependent="gdp_cap",
    unit_variable="country",
    time_variable="year",
    treatment_identifier="Sweden",
    controls_identifier=gdp_controls,
    time_optimize_ssr=range(1970, 1990),
)
synth_gdp = Synth()
synth_gdp.fit(dataprep=dp_gdp, optim_method="BFGS")

gdp_wide = gdp.pivot(index="year", columns="country", values="gdp_cap")
gdp_years = np.arange(1970, 2006)
w_gdp = synth_gdp.weights().reindex(gdp_controls).fillna(0)
gdp_actual = gdp_wide.loc[gdp_years, "Sweden"].values
gdp_synth = (gdp_wide.loc[gdp_years, gdp_controls] @ w_gdp).values

print("\nSynthetic-GDP donor weights (non-zero):")
print(w_gdp[w_gdp > 1e-4].sort_values(ascending=False).round(4))
print(f"GDP 2005 — Sweden actual: ${gdp_actual[-1]:,.0f} vs Synthetic: ${gdp_synth[-1]:,.0f}")

fig, ax = plt.subplots(figsize=(9, 5.4))
ax.plot(gdp_years, gdp_actual, color=WARM_ORANGE, lw=2.4, label="Sweden")
ax.plot(gdp_years, gdp_synth, color=STEEL_BLUE, lw=2.2, ls="--", label="Synthetic Sweden(GDP)")
ax.axvline(1990, color=LIGHT_TEXT, lw=0.8, ls=":")
ax.set_xlabel("Year")
ax.set_ylabel("GDP per capita (USD, real)")
ax.set_title("Synthetic GDP — did the carbon tax hurt Swedish growth?")
ax.legend(loc="lower right")
savefig("gdp_synth")

**Reading the output:**

- Synthetic GDP Sweden is dominated by Scandinavian peers — Denmark 61%, Norway 20%, Finland 10% — plus the US (9%).
- By 2005 actual GDP and synthetic GDP overlap to within **\$233 per capita** — less than 1% of the level.

No measurable growth penalty. Combined with the gap-plot evidence, GDP is **not** a confounder for the CO2 result. The policy worked *and* the economy was fine.

## 12. Tax incidence — did consumers really pay the tax?

So far the analysis has been *aggregate*. Now we zoom into the demand side to understand *why* consumers changed behaviour. First question: did the tax actually reach the consumer at the pump, or did oil companies absorb it?

We regress year-on-year **changes** in the retail price on changes in the oil price and changes in the total tax:

$$\Delta p^*_t = \beta_0 + \beta_1 \Delta \Theta_t + \beta_2 \Delta T_t + \varepsilon_t$$

The **pass-through coefficient** $\beta_2$ is the share of the tax change consumers actually pay. If $\beta_2 = 1$, consumers paid the full tax. If $\beta_2 = 0.5$, refiners absorbed half.

In [ ]:
reg = reg_data.copy()
tax_sub = reg[["year", "p_nom", "en_tax", "CO2_tax", "oil_p", "en_CO2_tax"]].copy()
tax_sub["delta_p"] = tax_sub["p_nom"].diff()
tax_sub["delta_oil_p"] = tax_sub["oil_p"].diff()
tax_sub["delta_tax"] = tax_sub["en_CO2_tax"].diff()
m_incid = pf.feols(
    "delta_p ~ delta_oil_p + delta_tax",
    data=tax_sub.dropna(subset=["delta_p", "delta_oil_p", "delta_tax"]),
    vcov="HC1",
)
print("\nTax-incidence regression (HC1):")
print(m_incid.tidy().round(4))

**Pass-through = 1.15** with SE 0.15. The 95% CI is roughly [0.85, 1.45], which contains 1.0. **We cannot reject full pass-through.**

This matters for everything below: when we estimate how much gasoline consumption fell in response to the tax, that response is to a real pump-price change.

## 13. OLS gasoline-consumption regressions (4 specs)

How strongly does Swedish demand respond to:

- A change in the **price excluding the carbon tax** ($pv_t$)?
- A change in the **carbon tax (including VAT)** ($ct_t$)?

If consumers are perfectly rational and only care about the total price, the two responses should be equal. If they differ, that tells us about behavioural channels (salience, permanence).

Andersson uses a **log-level** model:

$$\ln y_t = \beta_0 + \beta_1 pv_t + \beta_2 ct_t + \beta_3 D_t + \beta_4 X_t + \varepsilon_t$$

Because the outcome is in logs and regressors in levels, the coefficients are **semi-elasticities**: a one-SEK/litre rise in $x$ is associated with a $100\beta\%$ change in $y$. We estimate **four nested specs** (OLS1 to OLS4) adding one control at a time, with both HC1 and Newey–West HAC(16) standard errors.

In [ ]:
# pyfixest baseline (HC1)
ols_specs = {
    "OLS1": "log_gas_cons ~ p_real_vat + real_CO2_tax_vat + d_CO2_tax + t",
    "OLS2": "log_gas_cons ~ p_real_vat + real_CO2_tax_vat + d_CO2_tax + t + gdp_cap",
    "OLS3": "log_gas_cons ~ p_real_vat + real_CO2_tax_vat + d_CO2_tax + t + gdp_cap + urban_pop",
    "OLS4": "log_gas_cons ~ p_real_vat + real_CO2_tax_vat + d_CO2_tax + t + gdp_cap + urban_pop + unempl",
}
ols_fits = {name: pf.feols(f, data=reg, vcov="HC1") for name, f in ols_specs.items()}
for name, fit in ols_fits.items():
    print(f"\n{name} (HC1):")
    print(fit.tidy()[["Estimate", "Std. Error", "t value", "Pr(>|t|)"]].round(4))

# Newey-West HAC SE via statsmodels, to match Andersson's Stata `newey ... lag(16)`
def newey_west_table(formula: str, data: pd.DataFrame, maxlags: int = 16) -> pd.DataFrame:
    rhs = formula.split("~")[1]
    ys = formula.split("~")[0].strip()
    cols = [v.strip() for v in rhs.split("+")]
    sub = data[[ys] + cols].dropna()
    X = sm.add_constant(sub[cols])
    res = sm.OLS(sub[ys], X).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags})
    return pd.DataFrame({
        "coef": res.params, "se_nw16": res.bse,
        "t": res.tvalues, "p": res.pvalues,
    })

nw_tables = {name: newey_west_table(f, reg) for name, f in ols_specs.items()}
print("\nNewey-West HAC(16) results — OLS4:")
print(nw_tables["OLS4"].round(4))

# Save a wide comparison table
wide = pd.concat({k: v[["coef", "se_nw16"]] for k, v in nw_tables.items()}, axis=1)
wide.to_csv(ROOT / "tab_ols_newey_west.csv")

**OLS4 headline numbers:**

- **Price semi-elasticity:** β = −0.060 → a 1 SEK/litre rise in price → **6% lower** gasoline consumption.
- **Tax semi-elasticity:** β = −0.186 → a 1 SEK/litre rise in carbon tax → **18.6% lower** consumption.

The tax response is roughly **three times the price response**, stable across all four specifications. Both significant under both HC1 and Newey–West SEs.

## 14. Instrumental variables (2SLS) — addressing endogeneity

OLS gives the right answer **only if** the explanatory variables are uncorrelated with the regression's error term. If they are correlated, the coefficient is **biased** — that problem is called **endogeneity**.

In our setting, the carbon-tax-exclusive price $pv_t$ might be endogenous. An unobserved demand shock (push for electric vehicles, EU regulation tightening) could lower demand *and* shift the price. OLS would mis-attribute the demand shock to the price coefficient.

The fix is **two-stage least squares (2SLS)** using an **instrument** $z$ that satisfies:

1. **Relevance:** $z$ is correlated with $pv_t$.
2. **Exogeneity:** $z$ is *not* correlated with the error term.

Andersson proposes two instruments:

- **Real crude oil price** — exogenous because Sweden is too small to move world oil markets.
- **Real energy tax** — exogenous because tax changes are set by policy on long lead times.

In [ ]:
iv_data = reg[(reg["year"] >= 1970) & (reg["year"] <= 2011)].copy()

# Single-instrument: crude oil price (real)
iv2 = pf.feols(
    "log_gas_cons ~ real_CO2_tax_vat + d_CO2_tax + t + gdp_cap + urban_pop + unempl "
    "| p_real_vat ~ oil_p_real",
    data=iv_data, vcov="HC1",
)
print("\nIV: oil_p_real instrument (HC1):")
print(iv2.tidy().round(4))

# Single-instrument: energy tax
iv1 = pf.feols(
    "log_gas_cons ~ real_CO2_tax_vat + d_CO2_tax + t + gdp_cap + urban_pop + unempl "
    "| p_real_vat ~ real_en_tax_vat",
    data=iv_data, vcov="HC1",
)
print("\nIV: real_en_tax_vat instrument (HC1):")
print(iv1.tidy().round(4))

# Both instruments together (over-identified)
iv_both = pf.feols(
    "log_gas_cons ~ real_CO2_tax_vat + d_CO2_tax + t + gdp_cap + urban_pop + unempl "
    "| p_real_vat ~ oil_p_real + real_en_tax_vat",
    data=iv_data, vcov="HC1",
)
print("\nIV: both instruments (HC1):")
print(iv_both.tidy().round(4))

# First-stage diagnostic: how strong is each instrument?
fs_both = pf.feols(
    "p_real_vat ~ real_CO2_tax_vat + d_CO2_tax + t + gdp_cap + urban_pop + unempl + "
    "oil_p_real + real_en_tax_vat",
    data=iv_data, vcov="HC1",
)
print("\nFirst-stage regression (HC1):")
print(fs_both.tidy().round(4))


def coef_se(fit, var):
    t = fit.tidy()
    if var in t.index:
        row = t.loc[var]
        return float(row["Estimate"]), float(row["Std. Error"])
    return None, None


iv_summary = pd.DataFrame({
    "model": ["OLS4", "IV (energy tax)", "IV (oil price)", "IV (both)"],
    "beta_p_real_vat": [
        coef_se(ols_fits["OLS4"], "p_real_vat")[0],
        coef_se(iv1, "p_real_vat")[0],
        coef_se(iv2, "p_real_vat")[0],
        coef_se(iv_both, "p_real_vat")[0],
    ],
    "beta_real_CO2_tax_vat": [
        coef_se(ols_fits["OLS4"], "real_CO2_tax_vat")[0],
        coef_se(iv1, "real_CO2_tax_vat")[0],
        coef_se(iv2, "real_CO2_tax_vat")[0],
        coef_se(iv_both, "real_CO2_tax_vat")[0],
    ],
}).round(4)
iv_summary.to_csv(ROOT / "tab_iv_comparison.csv", index=False)
print("\nOLS vs IV comparison:")
print(iv_summary.to_string(index=False))

# Coefficient comparison plot
fig, ax = plt.subplots(figsize=(9, 5.4))
x_lbl = ["price semi-elasticity\n($p^v_t$)", "tax semi-elasticity\n($ct_t$)"]
x = np.arange(2)
bar_w = 0.2
models = iv_summary["model"].tolist()
colors_m = [STEEL_BLUE, WARM_ORANGE, TEAL, "#c179c8"]
for i, m in enumerate(models):
    row = iv_summary[iv_summary["model"] == m].iloc[0]
    ax.bar(x + (i - 1.5) * bar_w,
           [row["beta_p_real_vat"], row["beta_real_CO2_tax_vat"]],
           width=bar_w, color=colors_m[i], label=m)
ax.set_xticks(x)
ax.set_xticklabels(x_lbl)
ax.axhline(0, color=LIGHT_TEXT, lw=0.6)
ax.set_ylabel("Coefficient (log gasoline consumption per SEK / litre)")
ax.set_title("Price and tax semi-elasticities — OLS4 vs IV")
ax.legend(loc="lower right")
savefig("iv_vs_ols_coefs")

**Reading the output:**

Across all three IV specifications, the tax semi-elasticity is pinned to **−0.186** — identical to OLS4 to four decimal places. The price elasticity nudges from −0.060 (OLS) to −0.064 (IV with oil price).

This near-identical agreement is itself informative. If OLS had been badly biased, IV would have moved it noticeably. So we treat the OLS4 coefficients as **causal** price and tax elasticities.

### Why a 3× tax-vs-price asymmetry?

Two channels in the literature:

- **Salience:** a tax increase is *announced*. It appears in the news. A market price increase is a slow drift on the petrol-station billboard.
- **Permanence:** a tax increase is *persistent*. Market prices fluctuate; consumers may wait them out. A tax change triggers long-term decisions (vehicle purchase, commute distance).

Policy implication: revenue-neutral tax swaps (raise the carbon tax, cut something else) can deliver real emission reductions even when the average household's tax burden does not rise.

## 15. Disentangling carbon tax from VAT

The 1990/91 reform was a **bundle**: a new carbon tax, a new VAT on transport fuel, and a small cut to the pre-existing energy tax. The synthetic-control number captures the *total* bundle effect. What fraction is the carbon tax alone?

Andersson simulates the demand model under three counterfactual scenarios:

| Scenario | What's switched on | What's switched off |
| --- | --- | --- |
| `CarbonTaxandVAT` (actual) | All three components | Nothing |
| `NoCarbonTaxWithVAT` | VAT + energy tax | Carbon tax |
| `NoCarbonTaxNoVAT` | Energy tax only | Carbon tax + VAT |

The **vertical wedge between orange (actual) and blue-dashed (no carbon tax, with VAT)** is the carbon-tax-only contribution.

In [ ]:
dis = disent[(disent["year"] >= 1970) & (disent["year"] <= 2005)].copy()
print(dis.tail(6).round(4).to_string(index=False))

# Recompute Andersson's headline number: average carbon-tax-attributable reduction
# in CO2 transport emissions (relative to NoCarbonTax + Synthetic Sweden gap).
mask_post = dis["year"] >= 1990
dis_post = dis[mask_post]
ct_reduction_pct = (
    (dis_post["NoCarbonTaxWithVAT"] - dis_post["CarbonTaxandVAT"])
    / dis_post["NoCarbonTaxWithVAT"]
).mean() * 100
print(f"\nMean post-1990 carbon-tax-attributable reduction "
      f"(rel. to no-carbon-tax-with-VAT scenario): {ct_reduction_pct:.2f}%")

fig, ax = plt.subplots(figsize=(9, 5.4))
ax.plot(dis["year"], dis["NoCarbonTaxNoVAT"], color=TEAL, lw=2.2, ls=":",
        label="No carbon tax, no VAT")
ax.plot(dis["year"], dis["NoCarbonTaxWithVAT"], color=STEEL_BLUE, lw=2.2, ls="--",
        label="No carbon tax, with VAT")
ax.plot(dis["year"], dis["CarbonTaxandVAT"], color=WARM_ORANGE, lw=2.4,
        label="Carbon tax + VAT (actual)")
ax.axvline(1990, color=LIGHT_TEXT, lw=0.8, ls=":")
ax.set_xlabel("Year")
ax.set_ylabel("Metric tons CO$_2$ / capita (transport)")
ax.set_title("Counterfactual CO$_2$ scenarios — disentangling tax components")
ax.legend(loc="upper left")
savefig("disentangling")

dis.to_csv(ROOT / "tab_disentangling.csv", index=False)

**Reading the plot and numbers:**

- **2005 carbon-tax-only effect:** −0.57 t/capita, roughly **75% of the total reform wedge** in that year.
- **Average post-1990 carbon-tax-only reduction:** 9.5% (relative to no-carbon-tax-with-VAT baseline). Andersson, using the synthetic-Sweden baseline, reports the equivalent 6.3% with the same underlying ~0.17 t/capita wedge.

The carbon tax does **most** of the work after 2000, when the rate is ratcheted up sharply.

## 16. Headline summary — five numbers to remember

In [ ]:
summary = {
    "DiD (Sweden vs Denmark) on Sweden_post":
        m_did2.tidy().loc["Sweden_post", "Estimate"],
    "DiD (Sweden vs OECD, cluster SE) on Sweden_post":
        m_did_oecd.tidy().loc["Sweden_post", "Estimate"],
    "Synthetic Sweden — 2005 gap (t CO2/cap)":
        y_sweden.loc[2005] - y_synth.loc[2005],
    "Synthetic Sweden — average post-treatment % reduction":
        avg_post_pct,
    "Permutation placebo p-value (post/pre MSPE ratio)":
        p_val,
    "OLS4 — price semi-elasticity (beta1)":
        ols_fits["OLS4"].tidy().loc["p_real_vat", "Estimate"],
    "OLS4 — tax semi-elasticity (beta2)":
        ols_fits["OLS4"].tidy().loc["real_CO2_tax_vat", "Estimate"],
    "IV (oil) — price semi-elasticity":
        iv2.tidy().loc["p_real_vat", "Estimate"],
    "IV (oil) — tax semi-elasticity":
        iv2.tidy().loc["real_CO2_tax_vat", "Estimate"],
    "Disentangling — mean carbon-tax-only reduction (%)":
        ct_reduction_pct,
}
summary_df = pd.DataFrame({"value": summary}).round(4)
print(summary_df.to_string())
summary_df.to_csv(ROOT / "tab_headline_summary.csv")


print("\n=== Script completed successfully ===")

| Quantity | Value | What it means |
| --- | --- | --- |
| Synthetic Sweden — average gap | **−11.3%** per year (1990–2005) | The carbon tax cut transport CO2 by about a tenth, every year, for 16 years |
| Permutation p-value | **0.067** | Only 1 in 15 placebo countries shows a gap as big as Sweden's |
| Leave-one-out range | **8.8% to 13%** | Result survives dropping any single high-weight donor |
| Tax-vs-price asymmetry | **3×** | Consumers cut consumption 3× harder per SEK of tax than per SEK of price |
| Synthetic GDP gap | **< \$233 / capita** | No detectable growth penalty |

---

## Next steps

- Read the [full blog post](https://carlos-mendez.org/post/python_sc_co2tax/) for the deeper narrative.
- Try `pysyncon.AugSynth` (augmented synthetic control with ridge regularisation) or `pysyncon.PenalizedSynth`. Does the headline gap move?
- Extend the panel beyond 2005 with more recent OECD data.
- Replace the OLS/IV inference with `pyfixest`'s wild-cluster bootstrap.

## Three self-study exercises

1. **Sensitivity to the donor pool.** Drop Denmark from the donor list before fitting `pysyncon.Synth`. Does the post-1990 gap shrink, stay the same, or grow?
2. **Alternative predictors.** Re-fit Synthetic Sweden with only the four economic predictors and *no* lagged CO2 levels in `special_predictors`. Does the pre-treatment fit deteriorate?
3. **Augmented synthetic control.** Replace `Synth()` with `pysyncon.AugSynth()`. Compare the headline post-treatment gap and donor weights to the constrained-Synth solution.

## Citations

- Andersson, J. J. (2019). [Carbon Taxes and CO2 Emissions: Sweden as a Case Study](https://doi.org/10.1257/pol.20170144). *AEJ: Economic Policy* 11(4), 1–30.
- Graefe, T. (2020). [RTutorCarbonTaxesAndCO2Emissions](https://github.com/TheresaGraefe/RTutorCarbonTaxesAndCO2Emissions).
- Abadie, A., Diamond, A., & Hainmueller, J. (2010, 2015). Synthetic-control method papers in *JASA* and *AJPS*.